libraries

In [3]:

# Cell 2: Define tools

import ast
import operator
from datetime import datetime, timezone

def get_current_time():
    """Return the current UTC time."""
    return datetime.now(timezone.utc).isoformat()

def search_knowledge(query):
    """Search a small local knowledge base."""
    knowledge = {
        "agents": "An AI agent uses a model, tools, and a control loop.",
        "memory": "Agent memory stores information across steps or sessions.",
        "mcp": "MCP standardizes how AI applications connect to tools and resources.",
        "langgraph": "LangGraph represents workflows as stateful graphs.",
        "crewai": "CrewAI coordinates specialized agents and tasks."
    }

    query = query.lower()

    matches = [
        value for key, value in knowledge.items()
        if key in query or query in key
    ]

    return matches or ["No matching entry found."]

TOOLS = {
    "get_current_time": get_current_time,
    "search_knowledge": search_knowledge
}

print(TOOLS["search_knowledge"]("agents"))
print(TOOLS["get_current_time"]())


['An AI agent uses a model, tools, and a control loop.']
2026-10-28T18:24:07.773373+00:00


In [4]:

# Cell 3: Basic agent control loop

def basic_agent(task):
    task_lower = task.lower()
    observations = []

    if "time" in task_lower:
        result = TOOLS["get_current_time"]()
        observations.append({
            "tool": "get_current_time",
            "result": result
        })

    if any(word in task_lower for word in [
        "agent", "memory", "mcp", "langgraph", "crewai"
    ]):
        result = TOOLS["search_knowledge"](task)
        observations.append({
            "tool": "search_knowledge",
            "result": result
        })

    if not observations:
        observations.append({
            "tool": "none",
            "result": "No matching tool was selected."
        })

    return {
        "task": task,
        "observations": observations
    }


result = basic_agent(
    "Explain AI agents and tell me the current time."
)

print(result)


{'task': 'Explain AI agents and tell me the current time.', 'observations': [{'tool': 'get_current_time', 'result': '2026-10-28T18:24:27.340957+00:00'}, {'tool': 'search_knowledge', 'result': ['An AI agent uses a model, tools, and a control loop.']}]}


In [5]:

# Cell 4: Safe calculator

import ast
import operator

_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos
}

def safe_calculate(expression):
    """Evaluate basic arithmetic without using eval()."""

    def evaluate(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value

        if isinstance(node, ast.BinOp):
            op = _ALLOWED_OPERATORS.get(type(node.op))
            if op is None:
                raise ValueError("Operator not allowed")

            left = evaluate(node.left)
            right = evaluate(node.right)

            if isinstance(node.op, ast.Pow) and abs(right) > 10:
                raise ValueError("Exponent too large")

            result = op(left, right)

            if abs(result) > 1e12:
                raise ValueError("Result too large")

            return result

        if isinstance(node, ast.UnaryOp):
            op = _ALLOWED_OPERATORS.get(type(node.op))
            if op is None:
                raise ValueError("Unary operator not allowed")
            return op(evaluate(node.operand))

        raise ValueError("Unsupported expression")

    tree = ast.parse(expression, mode="eval")
    return evaluate(tree.body)


print(safe_calculate("(25 + 15) * 3"))


120


In [6]:

# Cell 5: Function-calling schema

function_schema = {
    "type": "function",
    "function": {
        "name": "safe_calculate",
        "description": "Calculate a basic arithmetic expression.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A basic arithmetic expression"
                }
            },
            "required": ["expression"],
            "additionalProperties": False
        },
        "strict": True
    }
}

print(function_schema)


{'type': 'function', 'function': {'name': 'safe_calculate', 'description': 'Calculate a basic arithmetic expression.', 'parameters': {'type': 'object', 'properties': {'expression': {'type': 'string', 'description': 'A basic arithmetic expression'}}, 'required': ['expression'], 'additionalProperties': False}, 'strict': True}}


In [7]:

# Cell 6: Execute a simulated function call

import json

simulated_model_call = {
    "name": "safe_calculate",
    "arguments": {
        "expression": "(1200 + 800) * 1.13"
    }
}

tool_name = simulated_model_call["name"]
arguments = simulated_model_call["arguments"]

if tool_name != "safe_calculate":
    raise ValueError("Unknown tool")

if set(arguments) != {"expression"}:
    raise ValueError("Unexpected arguments")

tool_result = safe_calculate(arguments["expression"])

print("Tool request:", json.dumps(simulated_model_call))
print("Tool result:", tool_result)


Tool request: {"name": "safe_calculate", "arguments": {"expression": "(1200 + 800) * 1.13"}}
Tool result: 2260.0


In [8]:

# Cell 7: Conversation memory

class ConversationMemory:
    def __init__(self, max_messages=10):
        self.messages = []
        self.max_messages = max_messages

    def add(self, role, content):
        self.messages.append({
            "role": role,
            "content": content
        })
        self.messages = self.messages[-self.max_messages:]

    def get_history(self):
        return list(self.messages)


memory = ConversationMemory(max_messages=6)

memory.add("user", "I am learning LLM agents.")
memory.add("assistant", "We will build agents step by step.")
memory.add("user", "Next, teach me MCP.")

print(memory.get_history())


[{'role': 'user', 'content': 'I am learning LLM agents.'}, {'role': 'assistant', 'content': 'We will build agents step by step.'}, {'role': 'user', 'content': 'Next, teach me MCP.'}]


In [9]:

# Cell 8: Save and reload long-term memory

import json
from pathlib import Path

MEMORY_FILE = Path("/content/agent_memory.json")

def save_memory(key, value):
    if MEMORY_FILE.exists():
        data = json.loads(MEMORY_FILE.read_text())
    else:
        data = {}

    data[key] = value
    MEMORY_FILE.write_text(json.dumps(data, indent=2))

def read_memory(key, default=None):
    if not MEMORY_FILE.exists():
        return default

    data = json.loads(MEMORY_FILE.read_text())
    return data.get(key, default)


save_memory(
    "learning_goal",
    "Become an LLM Engineer"
)

print(read_memory("learning_goal"))


Become an LLM Engineer


In [10]:

# Cell 9: Retrieve memory by semantic similarity

from sentence_transformers import SentenceTransformer
import numpy as np

memory_encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

long_term_facts = [
    "The learner wants to become an LLM Engineer.",
    "The learner is practicing Python agent tools.",
    "The learner wants to understand LangGraph workflows.",
    "The learner is learning model tool calling."
]

fact_vectors = memory_encoder.encode(
    long_term_facts,
    normalize_embeddings=True
)

def recall_memory(query, k=2):
    query_vector = memory_encoder.encode(
        [query],
        normalize_embeddings=True
    )

    scores = query_vector @ fact_vectors.T
    indices = np.argsort(scores[0])[::-1][:k]

    return [
        {
            "fact": long_term_facts[i],
            "similarity": float(scores[0][i])
        }
        for i in indices
    ]


for item in recall_memory(
    "What is my long-term AI career objective?"
):
    print(item)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{'fact': 'The learner is practicing Python agent tools.', 'similarity': 0.2743435502052307}
{'fact': 'The learner wants to become an LLM Engineer.', 'similarity': 0.2422633022069931}


In [11]:

# Cell 10: Planner -> Executor -> Evaluator

def planner(task):
    """Convert a task into a small ordered plan."""
    return [
        "Identify the task requirements",
        "Retrieve relevant information",
        "Prepare a response",
        "Evaluate the response"
    ]


def executor(task, plan):
    """Execute a simple deterministic workflow."""
    evidence = search_knowledge(task)

    return {
        "task": task,
        "plan": plan,
        "evidence": evidence,
        "draft": "Research summary: " + " ".join(evidence)
    }


def evaluator(result):
    """Return a simple quality check."""
    text = result["draft"]

    return {
        "passed": len(text.strip()) > 25,
        "length": len(text),
        "feedback": (
            "Draft meets the minimum length."
            if len(text.strip()) > 25
            else "Draft needs more evidence."
        )
    }


def run_workflow(task, max_retries=2):
    plan = planner(task)

    for attempt in range(max_retries + 1):
        result = executor(task, plan)
        evaluation = evaluator(result)

        print(f"Attempt {attempt + 1}: {evaluation}")

        if evaluation["passed"]:
            return {
                "plan": plan,
                "result": result,
                "evaluation": evaluation
            }

        plan.append("Improve the draft using additional evidence")

    return {
        "plan": plan,
        "result": result,
        "evaluation": evaluation
    }


workflow_result = run_workflow(
    "Explain MCP and LangGraph."
)

print("\nFINAL RESULT:")
print(workflow_result["result"]["draft"])


Attempt 1: {'passed': True, 'length': 137, 'feedback': 'Draft meets the minimum length.'}

FINAL RESULT:
Research summary: MCP standardizes how AI applications connect to tools and resources. LangGraph represents workflows as stateful graphs.


In [12]:

# Cell 11: Run a pretrained Hugging Face model

from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

samples = [
    "This agent framework is easy to understand.",
    "The application crashes every time I run it."
]

for sentence in samples:
    print(sentence)
    print(classifier(sentence))


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

This agent framework is easy to understand.
[{'label': 'POSITIVE', 'score': 0.9976231455802917}]
The application crashes every time I run it.
[{'label': 'NEGATIVE', 'score': 0.9997240900993347}]


In [13]:

# Cell 12: Define LangChain tools

from langchain_core.tools import tool

@tool
def calculate(expression: str) -> str:
    """Calculate a basic arithmetic expression."""
    return str(safe_calculate(expression))


@tool
def search_local_knowledge(query: str) -> str:
    """Search the local educational knowledge base."""
    return "\n".join(search_knowledge(query))


print(calculate.invoke({
    "expression": "15 * (8 + 2)"
}))

print(search_local_knowledge.invoke({
    "query": "MCP"
}))


150
MCP standardizes how AI applications connect to tools and resources.


In [14]:

# Cell 13: Optional real LangChain tool-calling agent

# In Colab, set OPENAI_API_KEY in the Secrets panel.
# Do not paste your API key into a notebook shared publicly.

import os

# Uncomment after setting your secret:
#
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
#
# %pip install -q -U langchain-openai
#
# from langchain_openai import ChatOpenAI
# from langchain.agents import create_agent
#
# llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
#
# agent = create_agent(
#     model=llm,
#     tools=[calculate, search_local_knowledge],
#     system_prompt=(
#         "You are a careful assistant. Use tools when needed. "
#         "Do not invent tool results."
#     )
# )
#
# response = agent.invoke({
#     "messages": [
#         {"role": "user",
#          "content": "Calculate 15 times 12 and explain MCP."}
#     ]
# })
#
# print(response["messages"][-1].content)


In [15]:

# Cell 14: Verify the MCP Python SDK

%pip -q install -U "mcp[cli]"

import mcp
print("MCP Python package loaded:", mcp.__name__)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.15.32 requires mcp<2,>=1.30.0, but you have mcp 2.3.0 which is incompatible.
MCP Python package loaded: mcp


In [1]:

# Cell 15: Create an MCP tool server

from mcp.server.fastmcp import FastMCP

mcp_server = FastMCP("LLM Agents Lab")

@mcp_server.tool()
def add_numbers(a: float, b: float) -> float:
    """Add two numbers and return their sum."""
    return a + b

@mcp_server.tool()
def lookup_topic(topic: str) -> str:
    """Look up a topic in the local agent knowledge base."""
    results = search_knowledge(topic)
    return "\n".join(results)

print("MCP server defined with add_numbers and lookup_topic.")


MCP server defined with add_numbers and lookup_topic.


In [4]:

# FIX: Define search_knowledge before running LangGraph

knowledge_base = [
    "MCP (Model Context Protocol) connects AI applications "
    "with external tools, resources, and services.",

    "Agent memory helps AI agents retain and retrieve "
    "information across tasks and conversations.",

    "Short-term memory stores current conversation context. "
    "Long-term memory stores information for future use.",

    "LangGraph builds stateful AI workflows using nodes, "
    "edges, shared state, and conditional routing."
]


def search_knowledge(query: str) -> list[str]:
    """Retrieve relevant documents using keyword matching."""
    query_words = set(query.lower().split())

    results = []

    for document in knowledge_base:
        document_words = set(document.lower().split())
        score = len(query_words & document_words)

        if score > 0:
            results.append((score, document))

    results.sort(key=lambda item: item[0], reverse=True)

    return [document for score, document in results[:3]]


# Verify the function exists and works
print("Function defined:", callable(search_knowledge))
print("Test results:", search_knowledge("Explain MCP and agent memory"))


Function defined: True
Test results: ['Agent memory helps AI agents retain and retrieve information across tasks and conversations.', 'MCP (Model Context Protocol) connects AI applications with external tools, resources, and services.', 'Short-term memory stores current conversation context. Long-term memory stores information for future use.']


In [5]:
final_state = agent_graph.invoke(initial_state)

print("ANSWER")
print(final_state["answer"])
print("Attempts:", final_state["attempts"])
print("Quality passed:", final_state["quality_ok"])

ANSWER
Task: Explain MCP and agent memory

Evidence-based summary:
Agent memory helps AI agents retain and retrieve information across tasks and conversations.
MCP (Model Context Protocol) connects AI applications with external tools, resources, and services.
Short-term memory stores current conversation context. Long-term memory stores information for future use.
Attempts: 1
Quality passed: True


In [6]:

# Cell 17: Multi-agent workflow without external API calls

def researcher_agent(topic):
    findings = search_knowledge(topic)

    return {
        "role": "Researcher",
        "findings": findings
    }


def writer_agent(research):
    content = "\n".join(research["findings"])

    return {
        "role": "Writer",
        "draft": (
            "Research summary\n\n"
            + content
        )
    }


def reviewer_agent(draft):
    text = draft["draft"]

    return {
        "role": "Reviewer",
        "approved": len(text) > 40,
        "feedback": (
            "Draft is sufficiently detailed."
            if len(text) > 40
            else "Draft needs additional content."
        )
    }


topic = "MCP and agents"

research = researcher_agent(topic)
draft = writer_agent(research)
review = reviewer_agent(draft)

print("RESEARCH:", research)
print("\nDRAFT:", draft["draft"])
print("\nREVIEW:", review)


RESEARCH: {'role': 'Researcher', 'findings': ['MCP (Model Context Protocol) connects AI applications with external tools, resources, and services.', 'Agent memory helps AI agents retain and retrieve information across tasks and conversations.', 'LangGraph builds stateful AI workflows using nodes, edges, shared state, and conditional routing.']}

DRAFT: Research summary

MCP (Model Context Protocol) connects AI applications with external tools, resources, and services.
Agent memory helps AI agents retain and retrieve information across tasks and conversations.
LangGraph builds stateful AI workflows using nodes, edges, shared state, and conditional routing.

REVIEW: {'role': 'Reviewer', 'approved': True, 'feedback': 'Draft is sufficiently detailed.'}


In [7]:
print("search_knowledge exists:",
      "search_knowledge" in globals())

print("research_node exists:",
      "research_node" in globals())

print("Current function:",
      globals().get("search_knowledge"))

search_knowledge exists: True
research_node exists: True
Current function: <function search_knowledge at 0x7ae416faef20>


In [8]:

# Cell 18: Configure a real CrewAI team (optional)

# Configure a supported model and API key before running.
# For example, set OPENAI_API_KEY through Colab Secrets.
#
# from crewai import Agent, Task, Crew, Process, LLM
#
# llm = LLM(
#     model="openai/gpt-4.1-mini",
#     temperature=0
# )
#
# researcher = Agent(
#     role="AI Researcher",
#     goal="Find accurate information about LLM agent architecture",
#     backstory="You research AI engineering concepts carefully.",
#     llm=llm,
#     verbose=True
# )
#
# writer = Agent(
#     role="Technical Writer",
#     goal="Explain technical ideas clearly for beginners",
#     backstory="You turn technical research into practical guides.",
#     llm=llm,
#     verbose=True
# )
#
# reviewer = Agent(
#     role="Quality Reviewer",
#     goal="Check accuracy, clarity, and unsupported claims",
#     backstory="You review technical content critically.",
#     llm=llm,
#     verbose=True
# )
#
# research_task = Task(
#     description="Research MCP, function calling, and agent memory.",
#     expected_output="A concise list of verified technical findings.",
#     agent=researcher
# )
#
# writing_task = Task(
#     description="Write a beginner-friendly explanation from the research.",
#     expected_output="A structured technical explanation.",
#     agent=writer,
#     context=[research_task]
# )
#
# review_task = Task(
#     description="Review the explanation for accuracy and missing details.",
#     expected_output="A review report and corrected final version.",
#     agent=reviewer,
#     context=[writing_task]
# )
#
# crew = Crew(
#     agents=[researcher, writer, reviewer],
#     tasks=[research_task, writing_task, review_task],
#     process=Process.sequential,
#     verbose=True
# )
#
# final_report = crew.kickoff()
# print(final_report)
